# Stage 05 - Compare and Select Models

Compare candidates on performance, explainability, stability, and governance requirements before selection.

In [ ]:
from pathlib import Path
import importlib.util
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILE = "readiness_observation_features.csv"
FEATURE_TABLE = "silver_readiness_feature_store"
DATA_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_FILE,
    Path("../data") / DATA_FILE,
    Path("data") / DATA_FILE,
    Path("Files") / DATA_FILE,
]


def locate_data_file(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    checked = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Checked: {checked}")


def min_max_scale(series):
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span

spark_session = globals().get("spark")
if spark_session is not None and spark_session.catalog.tableExists(FEATURE_TABLE):
    frame = spark_session.table(FEATURE_TABLE).toPandas()
    data_path = f"Lakehouse table {FEATURE_TABLE}"
else:
    data_path = locate_data_file(DATA_CANDIDATES)
    frame = pd.read_csv(data_path)

frame["feature_timestamp_utc"] = pd.to_datetime(frame["feature_timestamp_utc"], utc=True)
frame = frame.sort_values(
    ["scenario_id", "simulation_run_id", "system_instance_id", "feature_timestamp_utc"]
).reset_index(drop=True)
frame["quality_gap"] = 1.0 - frame["quality_rate"]
frame["feature_recency_minutes"] = frame["track_freshness_seconds"] / 60.0
frame["health_decline_flag"] = (frame["health_trend_index"] < 0).astype(int)
frame["maintenance_age_band_index"] = frame["maintenance_age_category"].map(
    {"fresh-service": 0, "steady-cycle": 1, "extended-cycle": 2}
).astype(int)
frame["run_quality_delta"] = frame["quality_rate"] - frame.groupby("simulation_run_id")["quality_rate"].transform("mean")
gap_totals = frame.groupby("simulation_run_id")["gap_count"].transform("sum").replace(0, 1)
frame["run_gap_share"] = frame["gap_count"] / gap_totals
frame["baseline_alignment_gap"] = frame["baseline_deviation_index"] + frame["quality_gap"]
frame["deterministic_baseline_score"] = (
    0.30 * min_max_scale(frame["track_freshness_seconds"])
    + 0.20 * min_max_scale(frame["gap_count"])
    + 0.20 * min_max_scale(frame["quality_gap"])
    + 0.15 * min_max_scale(frame["abstract_ack_lag_seconds"])
    + 0.10 * min_max_scale(frame["baseline_deviation_index"])
    + 0.05 * min_max_scale(frame["maintenance_age_days"])
)

feature_columns = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'maintenance_age_category', 'test_phase', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
numeric_features = ['track_freshness_seconds', 'gap_count', 'quality_rate', 'abstract_ack_lag_seconds', 'health_trend_index', 'maintenance_age_days', 'baseline_deviation_index', 'late_event_count', 'feature_recency_minutes', 'quality_gap', 'health_decline_flag', 'maintenance_age_band_index', 'run_quality_delta', 'run_gap_share', 'baseline_alignment_gap']
lineage_columns = ['classification', 'scenario_id', 'simulation_run_id', 'test_event_id', 'track_id', 'site_id', 'system_instance_id', 'system_family', 'feature_timestamp_utc', 'source_snapshot_id', 'feature_snapshot_id', 'baseline_snapshot_id', 'finding_snapshot_id']
categorical_features = ["maintenance_age_category", "test_phase"]
run_order = frame.groupby("simulation_run_id")["feature_timestamp_utc"].min().sort_values().index.tolist()
train_runs = run_order[:2]
validation_runs = run_order[2:3]
test_runs = run_order[3:]
train_frame = frame[frame["simulation_run_id"].isin(train_runs)].copy()
validation_frame = frame[frame["simulation_run_id"].isin(validation_runs)].copy()
test_frame = frame[frame["simulation_run_id"].isin(test_runs)].copy()
assert not set(feature_columns) & set(lineage_columns)


def prevalence_rank(probabilities, positive_rate):
    top_n = max(1, int(round(len(probabilities) * positive_rate)))
    ranking = pd.Series(probabilities).rank(method="first", ascending=False)
    return (ranking <= top_n).astype(int)


def metric_row(model_name, split_name, y_true, probabilities, positive_rate):
    y_true = pd.Series(y_true).astype(int)
    predicted = prevalence_rank(probabilities, positive_rate)
    return {
        "model_name": model_name,
        "split": split_name,
        "roc_auc": round(float(roc_auc_score(y_true, probabilities)), 4),
        "average_precision": round(float(average_precision_score(y_true, probabilities)), 4),
        "brier_loss": round(float(brier_score_loss(y_true, probabilities)), 4),
        "predicted_priority_rate": round(float(predicted.mean()), 4),
    }


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical_features,
        ),
    ]
)

candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=7),
    "random_forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=7),
}
if importlib.util.find_spec("lightgbm") is not None:
    from lightgbm import LGBMClassifier

    candidate_models["lightgbm"] = LGBMClassifier(
        n_estimators=120,
        learning_rate=0.05,
        num_leaves=15,
        random_state=7,
    )
else:
    candidate_models["hist_gradient_boosting_fallback"] = HistGradientBoostingClassifier(
        max_depth=4,
        learning_rate=0.08,
        max_iter=160,
        random_state=7,
    )


In [ ]:
comparison_rows = []
fitted_models = {}
train_positive_rate = float(train_frame["synthetic_review_priority_label"].mean())
for split_name, split_frame in [("validation", validation_frame), ("test", test_frame)]:
    comparison_rows.append(
        metric_row(
            "deterministic_baseline",
            split_name,
            split_frame["synthetic_review_priority_label"],
            split_frame["deterministic_baseline_score"],
            train_positive_rate,
        )
    )

for model_name, estimator in candidate_models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(train_frame[feature_columns], train_frame["synthetic_review_priority_label"])
    fitted_models[model_name] = pipeline
    for split_name, split_frame in [("validation", validation_frame), ("test", test_frame)]:
        probabilities = pipeline.predict_proba(split_frame[feature_columns])[:, 1]
        comparison_rows.append(
            metric_row(
                model_name,
                split_name,
                split_frame["synthetic_review_priority_label"],
                probabilities,
                train_positive_rate,
            )
        )

comparison_frame = pd.DataFrame(comparison_rows)
validation_ranking = comparison_frame[comparison_frame["split"] == "validation"].sort_values("roc_auc", ascending=False).reset_index(drop=True)
baseline_validation_auc = validation_ranking[validation_ranking["model_name"] == "deterministic_baseline"].iloc[0]["roc_auc"]
best_learned = validation_ranking[validation_ranking["model_name"] != "deterministic_baseline"].iloc[0]
DEMO_SELECTION_MARGIN = 0.02
if best_learned["roc_auc"] >= baseline_validation_auc + DEMO_SELECTION_MARGIN:
    selected_model_name = best_learned["model_name"]
else:
    selected_model_name = "deterministic_baseline"
selected_model = fitted_models.get(selected_model_name)

comparison_frame.sort_values(["split", "roc_auc"], ascending=[True, False]).reset_index(drop=True)


In [ ]:
invented_weight_table = pd.DataFrame(
    {
        "feature": [
            "track_freshness_seconds",
            "gap_count",
            "quality_gap",
            "abstract_ack_lag_seconds",
            "baseline_deviation_index",
            "maintenance_age_days",
        ],
        "importance": [0.30, 0.20, 0.20, 0.15, 0.10, 0.05],
    }
)
SHAP_AVAILABLE = importlib.util.find_spec("shap") is not None
if selected_model_name == "deterministic_baseline":
    explainability_frame = invented_weight_table.copy()
    explainability_note = "Selected the deterministic baseline, so the baseline weight table is the explanation output."
elif SHAP_AVAILABLE:
    import shap

    transformed = selected_model.named_steps["preprocessor"].transform(test_frame[feature_columns])
    transformed_columns = selected_model.named_steps["preprocessor"].get_feature_names_out()
    transformed_frame = pd.DataFrame(transformed, columns=transformed_columns, index=test_frame.index)
    explainer = shap.Explainer(selected_model.named_steps["model"], transformed_frame)
    shap_values = explainer(transformed_frame)
    shap_array = shap_values.values
    if getattr(shap_array, "ndim", 0) == 3:
        shap_array = shap_array[:, :, 1]
    explainability_frame = pd.DataFrame(
        {
            "feature": transformed_columns,
            "importance": np.abs(shap_array).mean(axis=0),
        }
    ).sort_values("importance", ascending=False).reset_index(drop=True)
    explainability_note = "Used mean absolute SHAP values because the dependency is available."
else:
    importance = permutation_importance(
        selected_model,
        test_frame[feature_columns],
        test_frame["synthetic_review_priority_label"],
        n_repeats=20,
        random_state=7,
    )
    explainability_frame = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": importance.importances_mean,
        }
    ).sort_values("importance", ascending=False).reset_index(drop=True)
    explainability_note = "SHAP is not available in this runtime, so permutation importance is used instead."

explainability_frame.head(10)


In [ ]:
model_validation_summary = {
    "classification": "SYNTHETIC_UNCLASS",
    "data_path": str(data_path),
    "split_strategy": {
        "group_key": "simulation_run_id",
        "train_runs": train_runs,
        "validation_runs": validation_runs,
        "test_runs": test_runs,
    },
    "selection_rule": f"Select a learned model only if validation ROC AUC exceeds the deterministic baseline by at least {DEMO_SELECTION_MARGIN:.2f}. This margin is invented for the demo.",
    "selected_model_name": selected_model_name,
    "leakage_controls": [
        "All lineage identifiers and snapshot identifiers are excluded from feature_columns.",
        "The label and any label-shaped names are excluded before fitting.",
        "The split boundary is the full simulation run, not random rows.",
    ],
    "comparison_metrics": comparison_frame.to_dict(orient="records"),
    "explainability_note": explainability_note,
    "lineage": {
        "source_snapshot_ids": sorted(frame["source_snapshot_id"].unique().tolist()),
        "feature_snapshot_ids": sorted(frame["feature_snapshot_id"].unique().tolist()),
        "baseline_snapshot_ids": sorted(frame["baseline_snapshot_id"].unique().tolist()),
        "finding_snapshot_ids": sorted(frame["finding_snapshot_id"].unique().tolist()),
    },
    "limitations": [
        "The sample is intentionally compact and synthetic.",
        "The selected model, if any, supports analyst review prioritization only.",
        "No official requirement status or readiness certification is created here.",
    ],
}

print(json.dumps(model_validation_summary, indent=2))
comparison_frame.sort_values(["split", "roc_auc"], ascending=[True, False]).reset_index(drop=True)
